# ChurnCast: who will cancel next month, and why?

The Python track's report for TiffinBox's retention team. Every number below is computed by the `churncast` package from the raw export, so re-running this notebook from a clean checkout reproduces it exactly (CI executes it with `make report`). The rules are in `contract/feature-spec.md`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from churncast.artefact import contributions, score
from churncast.pipeline import analyse

DATA = Path(os.environ.get("CHURNCAST_DATA", "../../../data/raw/2026-07"))  # relative to notebooks/
r = analyse(DATA)
m, f, model = r["metrics"], r["features"], r["artefact"]
print(m["dataset"], "| champion", m["champion"]["experiment"], "| gate", m["gate"]["passed"])

## 1 · Can we trust the export?
What cleaning removed or repaired, and the contract checks (an `error` check would have stopped the run before this cell).

In [ ]:
display(pd.DataFrame({"raw": m["data_quality"]["raw_rows"], "clean": m["data_quality"]["clean_rows"]}))
display(pd.Series(m["data_quality"]["issues"], name="rows").to_frame())
pd.DataFrame(m["data_quality"]["checks"])[["id", "severity", "status", "failing_rows"]]

## 2 · The timeline: one row per customer and month-end snapshot
Features use only what was known on the last day of month M; the label is whether the customer paid for month M+1. Training snapshots come first, the test months after them: the model is tested on the future, as it will be used.

In [ ]:
s = pd.DataFrame(m["snapshots"])
display(s)
colours = ["#9aa5b1" if x == "train" else "#d64545" for x in s["split"]]
ax = s.plot.bar(x="snapshot", y="churn_rate", color=colours, legend=False, figsize=(8, 3))
ax.set_title("Monthly churn rate (grey: training snapshots, red: test)")
ax.set_ylabel("churned next month")
plt.xticks(rotation=0)
plt.show()

## 3 · Leakage: too good to be true?
The single-feature AUC of every model feature on the training rows, and of the column the brief warned about (`cancellation_reason`, filled in after a customer leaves).

In [ ]:
leak = m["leakage"]
print(
    f"best feature {leak['max_feature']}: {leak['max_feature_auc']:.3f}; "
    f"cancellation_reason alone: {leak['probe_auc']:.3f} -> never a feature"
)
pd.Series(leak["feature_auc"]).sort_values().plot.barh(figsize=(7, 4), color="#9aa5b1")
plt.axvline(leak["probe_auc"], color="#d64545", ls="--", label="cancellation_reason (leaky)")
plt.legend()
plt.title("Single-feature AUC on the training rows")
plt.show()

## 4 · Baselines first, then the model
5-fold cross-validation grouped by customer (a customer never sits in two folds) chooses the champion; the test months are looked at once, at the end.

In [ ]:
rows = [
    {
        "id": e["id"],
        "name": e["name"],
        "cv_auc": e["cv"]["auc_mean"],
        "cv_sd": e["cv"]["auc_sd"],
        "test_auc": e["test"]["auc"],
        "test_brier": e["test"]["brier"],
        "precision@10%": e["test"]["precision_at_top"],
        "net ₹/month": e["test"]["net_value_inr_per_month"],
    }
    for e in m["experiments"]
]
display(pd.DataFrame(rows).set_index("id").round(4))
print(m["champion"]["reason"])

## 5 · Is a predicted 20 % really 20 %? (calibration)

In [ ]:
cal = pd.DataFrame(m["calibration"]["bins"]).query("n > 0")
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0, 1], [0, 1], ls="--", color="#9aa5b1")
ax.scatter(cal["mean_pred"], cal["observed"], s=cal["n"] / 10, color="#d64545")
ax.set(xlabel="mean predicted", ylabel="observed churn", xlim=(0, 1), ylim=(0, 1))
ax.set_title(f"Calibration on the test months (ECE {m['calibration']['ece']:.3f})")
plt.show()

## 6 · What the retention team gets: the top 10 % each month

In [ ]:
t = m["test"]
print(
    f"{t['contacted']} calls reach {t['churners_reached']} churners: precision "
    f"{t['precision_at_top']:.1%}, recall {t['recall_at_top']:.1%}, lift {t['lift_at_top']:.1f}x, "
    f"≈ ₹{t['net_value_inr_per_month']:,.0f} a month (assumptions in the contract)"
)

## 7 · Where does it work less well? Slices and fairness

In [ ]:
cols = ["n", "churn_rate", "mean_pred", "auc", "flagged_rate", "recall_at_top"]
for dim, rows in m["slices"].items():
    display(pd.DataFrame(rows).set_index("value")[cols].round(3).rename_axis(dim))
print(m["fairness"])

## 8 · Drivers and one explanation
Standardised coefficients (log-odds per standard deviation). For a single customer, the contribution of a feature is its coefficient times the distance from the training mean.

In [ ]:
display(pd.DataFrame(m["champion"]["importance"]).set_index("feature").round(4))
row = f[f["split"] == "test"].iloc[0][model["features"]].to_dict()
print(f"churn probability {score(model, row):.1%}")
pd.DataFrame(contributions(model, row)[:5])

## 9 · Recommendation


In [ ]:
g = m["gate"]
print("release gate:", "PASSED" if g["passed"] else "FAILED")
display(pd.DataFrame(g["checks"]).set_index("id"))

**Recommendation.** Ship the champion to the retention team as a ranked call list with reasons, and measure the save rate with a randomised holdout (call 90 % of the top decile, hold back 10 %): the model predicts *who* is at risk, not whether a call changes their mind. **Caveats:** observational data; acquisition channel and age band are not features; retrain when monitoring shows drift.

## 10 · v1.1: monitoring the deployed model on the new months
PSI of every feature against the training months (0.10: watch, 0.25: drift), then the realised performance of model 1.0.0 on the labelled months after deployment.

In [ ]:
mon = m["monitoring"]
display(pd.DataFrame(mon["drift"]).drop(columns=["feature_psi"]).set_index("snapshot"))
psi = pd.DataFrame({s["snapshot"]: s["feature_psi"] for s in mon["drift"]})
psi.plot.barh(figsize=(7, 5))
plt.axvline(0.1, ls="--", color="#9aa5b1")
plt.axvline(0.25, ls="--", color="#d64545")
plt.title("Population stability index per feature vs the training months")
plt.show()
display(pd.DataFrame(mon["performance"]).set_index("snapshot").drop(columns=["alerts"]).round(4))
print("status:", mon["status"])
for a in mon["alerts"]:
    print(" -", a)

## 11 · v1.1: champion vs challenger on the holdout month

In [ ]:
rt = m["retraining"]
ch = rt["challenger"]
span = f"{ch['refit_snapshots'][0]} to {ch['refit_snapshots'][-1]}"
print(span, "| calibrated on", ch["calibration_snapshot"], "| offset", round(ch["calibration_offset"], 4))
keys = ["auc", "brier", "ece", "mean_pred", "observed_rate", "precision_at_top", "recall_gap"]
both = pd.DataFrame({"champion 1.0.0": rt["champion_holdout"], "challenger": rt["challenger_holdout"]})
display(both.loc[keys].round(4))
display(pd.DataFrame(rt["gate"]["checks"]).set_index("id"))
print("decision:", rt["decision"])

**v1.1 recommendation.** The June price change moved `price_change_pct` far outside its training range and churn rose; model 1.0.0 still ranks well but under-predicts churn. Promote the recalibrated challenger as 1.1.0 and keep the monthly PSI and calibration checks.